05_clustering.ipynb

1. Подготовка выборки

2. Загрузка эмбеддингов

3. Проверка размерности

4. K-Means

5. Подбор количества кластеров

6. Silhouette Score

7. Размеры кластеров

8. Визуализация кластеров

9. HDBSCAN

10. Сравнение методов

11. Вывод

# Кластеризация эмбеддингов

Цель данного этапа — исследовать структуру векторных представлений товаров с помощью алгоритмов кластеризации.

В отличие от предварительной визуализации, выполненной в предыдущем ноутбуке, здесь необходимо использовать эмбеддинги, непосредственно связанные с товарами исходного датасета.

Для каждого объекта требуется сохранить соответствие между векторным представлением и его `category_id`. Это позволит не только обнаружить структуру кластеров, но и сравнить полученные группы с исходной категоризацией товаров.

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATA_PATH = "../data/raw/dataset-electronics-5M.parquet"

parquet_file = pq.ParquetFile(DATA_PATH)

print("Количество строк:", parquet_file.metadata.num_rows)
print("Количество row groups:", parquet_file.num_row_groups)
print("Колонки:", parquet_file.schema.names)

: 

In [ ]:
row_group = parquet_file.read_row_group(
    0,
    columns=["embedding", "category_id", "model_id"]
).to_pandas()

print(row_group.shape)
print(row_group.head())
print()
print("Тип embedding:", type(row_group["embedding"].iloc[0]))
print("Размер embedding:", len(row_group["embedding"].iloc[0]), "байт")

In [ ]:
import struct
import numpy as np

emb = row_group["embedding"].iloc[0]

print("Размер:", len(emb))
print("\nПервые 64 байта:")
print(emb[:64])

print("\nHEX:")
print(emb[:64].hex())

print("\nПервые 8 байт как uint64:")
print(struct.unpack("<Q", emb[:8]))

print("\nПервые 16 байт как float32:")
print(np.frombuffer(emb[:16], dtype="<f4"))

print("\nПервые 16 байт как float64:")
print(np.frombuffer(emb[:16], dtype="<f8"))

In [ ]:
for i in range(5):
    emb = row_group["embedding"].iloc[i]
    
    print(
        i,
        "size =", len(emb),
        "first_16 =", emb[:16].hex()
    )

In [ ]:
import numpy as np


def decode_embedding(data):
    """
    Декодирует embedding из бинарного формата Parquet.

    Формат:
    2 байта marker + 312 раз:
    8 байт float64 + 2 байта marker
    """
    
    data = bytes(data)
    
    # 3122 = 2 + 312 * 10
    if len(data) != 3122:
        raise ValueError(f"Неожиданный размер embedding: {len(data)}")
    
    values = np.empty(312, dtype=np.float64)
    
    for i in range(312):
        start = 2 + i * 10
        values[i] = np.frombuffer(
            data[start:start + 8],
            dtype="<f8"
        )[0]
    
    return values.astype(np.float32)

In [ ]:
emb = decode_embedding(row_group["embedding"].iloc[0])

print("Shape:", emb.shape)
print("Dtype:", emb.dtype)
print("First 10 values:", emb[:10])
print("Min:", emb.min())
print("Max:", emb.max())
print("Mean:", emb.mean())
print("Std:", emb.std())

In [ ]:
X_test = np.vstack([
    decode_embedding(x)
    for x in row_group["embedding"]
])

print("Shape:", X_test.shape)
print("Dtype:", X_test.dtype)
print("Min:", X_test.min())
print("Max:", X_test.max())
print("Mean:", X_test.mean())
print("Std:", X_test.std())

In [ ]:
norms = np.linalg.norm(X_test, axis=1)

print("Norm min:", norms.min())
print("Norm max:", norms.max())
print("Norm mean:", norms.mean())
print("Norm median:", np.median(norms))
print("Zero vectors:", np.sum(norms < 1e-6))

In [ ]:
# Нормы всех векторов
norms = np.linalg.norm(X_test, axis=1)

print("Нормы:")
print("Min:", norms.min())
print("Max:", norms.max())
print("Mean:", norms.mean())
print("Median:", np.median(norms))

print("\nНулевые векторы:", np.sum(norms < 1e-6))

# Проверка NaN / Inf
print("\nNaN:", np.isnan(X_test).sum())
print("Inf:", np.isinf(X_test).sum())


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(X_test[:100])

# Убираем диагональ
mask = ~np.eye(100, dtype=bool)

print("Средняя cosine similarity:", similarity[mask].mean())
print("Медиана:", np.median(similarity[mask]))
print("Min:", similarity[mask].min())
print("Max:", similarity[mask].max())

In [ ]:
categories = row_group["category_id"].to_numpy()

unique_categories, counts = np.unique(
    categories,
    return_counts=True
)

print("Категории в row group:")
for category, count in zip(unique_categories, counts):
    print(category, "->", count)

In [ ]:
target_category = unique_categories[np.argmax(counts)]

idx = np.where(categories == target_category)[0]

X_category = X_test[idx]

similarity_category = cosine_similarity(X_category[:100])

mask = ~np.eye(len(X_category[:100]), dtype=bool)

print("Категория:", target_category)
print("Количество товаров:", len(X_category))

print(
    "Средняя similarity:",
    similarity_category[mask].mean()
)

In [ ]:
SAMPLE_SIZE = 100_000
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)

total_rows = parquet_file.metadata.num_rows

sample_indices = np.sort(
    rng.choice(
        total_rows,
        size=SAMPLE_SIZE,
        replace=False
    )
)

print("Всего строк:", total_rows)
print("Размер выборки:", len(sample_indices))

In [ ]:
X_list = []
y_list = []

collected = 0

for rg in range(parquet_file.num_row_groups):
    
    if collected >= SAMPLE_SIZE:
        break
    
    batch = parquet_file.read_row_group(
        rg,
        columns=["embedding", "category_id"]
    ).to_pandas()
    
    remaining = SAMPLE_SIZE - collected
    
    if len(batch) > remaining:
        batch = batch.iloc[:remaining]
    
    X_batch = np.vstack([
        decode_embedding(x)
        for x in batch["embedding"]
    ])
    
    y_batch = batch["category_id"].to_numpy()
    
    X_list.append(X_batch)
    y_list.append(y_batch)
    
    collected += len(batch)
    
    if rg % 100 == 0:
        print(
            f"row group: {rg}, "
            f"собрано: {collected:,}"
        )

X = np.vstack(X_list)
y = np.concatenate(y_list)

print("\nИтог:")
print("X:", X.shape)
print("y:", y.shape)

In [ ]:
unique_categories, category_counts = np.unique(
    y,
    return_counts=True
)

print("Количество категорий:", len(unique_categories))
print("Минимальный размер категории:", category_counts.min())
print("Максимальный размер категории:", category_counts.max())
print("Медианный размер категории:", np.median(category_counts))

In [ ]:
top_idx = np.argsort(category_counts)[-10:][::-1]

for i in top_idx:
    print(
        f"category_id={unique_categories[i]}: "
        f"{category_counts[i]:,}"
    )

In [ ]:
from sklearn.cluster import KMeans

k_values = [5, 10, 25, 50]

kmeans_results = {}

for k in k_values:
    print(f"\nK = {k}")
    
    kmeans = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    
    labels = kmeans.fit_predict(X)
    
    kmeans_results[k] = {
        "model": kmeans,
        "labels": labels
    }
    
    print("Готово")
    print("Размеры кластеров:")
    print(np.bincount(labels))

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import normalized_mutual_info_score

results = []

for k, result in kmeans_results.items():
    
    labels = result["labels"]
    
    # Silhouette на подвыборке,
    # чтобы не тратить слишком много памяти/времени
    rng = np.random.default_rng(RANDOM_STATE)
    
    silhouette_idx = rng.choice(
        len(X),
        size=min(20_000, len(X)),
        replace=False
    )
    
    silhouette = silhouette_score(
        X[silhouette_idx],
        labels[silhouette_idx],
        metric="cosine"
    )
    
    ari = adjusted_rand_score(y, labels)
    
    nmi = normalized_mutual_info_score(y, labels)
    
    results.append({
        "K": k,
        "Silhouette": silhouette,
        "ARI": ari,
        "NMI": nmi
    })

results_df = pd.DataFrame(results)

results_df

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Avenir Next"

ACCENT = "#6C63FF"

fig, ax = plt.subplots(figsize=(11, 5.5))

metrics = [
    ("Silhouette", "-", "o"),
    ("ARI", "--", "s"),
    ("NMI", ":", "^")
]

for metric, linestyle, marker in metrics:
    ax.plot(
        results_df["K"],
        results_df[metric],
        color=ACCENT,
        linestyle=linestyle,
        marker=marker,
        linewidth=2.2,
        markersize=6,
        label=metric
    )

    for _, row in results_df.iterrows():
        value = row[metric]

        ax.annotate(
            f"{value:.3f}",
            (row["K"], value),
            xytext=(0, 9),
            textcoords="offset points",
            ha="center",
            fontsize=9
        )

ax.set_title(
    "Качество кластеризации при разных K",
    fontsize=20,
    fontweight="bold",
    pad=25
)

ax.set_xlabel(
    "Количество кластеров",
    fontsize=11,
    labelpad=12
)

ax.set_ylabel(
    "Значение метрики",
    fontsize=11,
    labelpad=12
)

ax.set_xticks(results_df["K"])

ax.grid(
    axis="y",
    alpha=0.12,
    linewidth=0.8
)

ax.tick_params(
    axis="both",
    length=0,
    labelsize=10
)

ax.legend(
    frameon=False,
    loc="upper right"
)

# Убираем рамку полностью
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

Мы получили только 5 категорий, хотя во всей выборке EDA было 122 категории. Причина в том, что текущая загрузка берёт первые 100 000 товаров, а Parquet, судя по всему, организован неслучайно — первые row groups сильно сконцентрированы на нескольких категориях.

То есть сейчас это не репрезентативная случайная выборка.

Поэтому я бы не записывал пока K=5 как окончательный результат.

In [ ]:
SAMPLE_SIZE = 100_000
RANDOM_STATE = 42

# Сначала читаем только category_id
category_data = parquet_file.read(
    columns=["category_id"]
).to_pandas()

print("Всего строк:", len(category_data))
print("Категорий:", category_data["category_id"].nunique())

In [ ]:
sample_fraction = SAMPLE_SIZE / len(category_data)

sample_indices = (
    category_data
    .groupby("category_id", group_keys=False)
    .sample(
        frac=sample_fraction,
        random_state=RANDOM_STATE
    )
    .index
)

sample_indices = np.sort(sample_indices.to_numpy())

print("Размер выборки:", len(sample_indices))

In [ ]:
sample_categories = category_data.iloc[sample_indices]["category_id"]

print(
    "Категорий в выборке:",
    sample_categories.nunique()
)

print(
    sample_categories.value_counts().describe()
)

In [ ]:
sample_indices_set = set(sample_indices)

X_list = []
y_list = []

current_start = 0

for rg in range(parquet_file.num_row_groups):

    metadata = parquet_file.metadata.row_group(rg)
    num_rows = metadata.num_rows

    current_end = current_start + num_rows

    # Индексы выборки внутри текущего row group
    local_indices = [
        idx - current_start
        for idx in sample_indices
        if current_start <= idx < current_end
    ]

    if local_indices:

        batch = parquet_file.read_row_group(
            rg,
            columns=["embedding", "category_id"]
        ).to_pandas()

        selected = batch.iloc[local_indices]

        X_batch = np.vstack([
            decode_embedding(x)
            for x in selected["embedding"]
        ])

        y_batch = selected["category_id"].to_numpy()

        X_list.append(X_batch)
        y_list.append(y_batch)

    current_start = current_end

    if rg % 200 == 0:
        print(
            f"Обработано row groups: {rg}/{parquet_file.num_row_groups}"
        )

X = np.vstack(X_list)
y = np.concatenate(y_list)

print("\nИтог:")
print("X:", X.shape)
print("y:", y.shape)
print("Категорий:", len(np.unique(y)))

In [ ]:
from sklearn.cluster import KMeans

k_values = [5, 10, 25, 50, 100]

kmeans_results = {}

for k in k_values:
    print(f"\nK = {k}")
    
    kmeans = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    
    labels = kmeans.fit_predict(X)
    
    kmeans_results[k] = {
        "model": kmeans,
        "labels": labels
    }
    
    print("Готово")
    print("Размеры кластеров:")
    print(np.bincount(labels))

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import normalized_mutual_info_score

results = []

for k, result in kmeans_results.items():
    
    labels = result["labels"]
    
    # Silhouette на подвыборке,
    # чтобы не тратить слишком много памяти/времени
    rng = np.random.default_rng(RANDOM_STATE)
    
    silhouette_idx = rng.choice(
        len(X),
        size=min(20_000, len(X)),
        replace=False
    )
    
    silhouette = silhouette_score(
        X[silhouette_idx],
        labels[silhouette_idx],
        metric="cosine"
    )
    
    ari = adjusted_rand_score(y, labels)
    
    nmi = normalized_mutual_info_score(y, labels)
    
    results.append({
        "K": k,
        "Silhouette": silhouette,
        "ARI": ari,
        "NMI": nmi
    })

results_df = pd.DataFrame(results)

results_df

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Avenir Next"

ACCENT = "#6C63FF"

fig, ax = plt.subplots(figsize=(11, 5.5))

metrics = [
    ("Silhouette", "-", "o"),
    ("ARI", "--", "s"),
    ("NMI", ":", "^")
]

for metric, linestyle, marker in metrics:
    ax.plot(
        results_df["K"],
        results_df[metric],
        color=ACCENT,
        linestyle=linestyle,
        marker=marker,
        linewidth=2.2,
        markersize=6,
        label=metric
    )

    for _, row in results_df.iterrows():
        value = row[metric]

        ax.annotate(
            f"{value:.3f}",
            (row["K"], value),
            xytext=(0, 9),
            textcoords="offset points",
            ha="center",
            fontsize=9
        )

ax.set_title(
    "Качество кластеризации при разных K",
    fontsize=20,
    fontweight="bold",
    pad=25
)

ax.set_xlabel(
    "Количество кластеров",
    fontsize=11,
    labelpad=12
)

ax.set_ylabel(
    "Значение метрики",
    fontsize=11,
    labelpad=12
)

ax.set_xticks(results_df["K"])

ax.grid(
    axis="y",
    alpha=0.12,
    linewidth=0.8
)

ax.tick_params(
    axis="both",
    length=0,
    labelsize=10
)

ax.legend(
    frameon=False,
    loc="upper right"
)

# Убираем рамку полностью
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
k_values_detailed = [5, 10, 15, 20, 25, 30, 40, 50, 75, 100]

results_detailed = []

rng = np.random.default_rng(42)

# Одна и та же подвыборка для Silhouette,
# чтобы сравнение K было честным
silhouette_idx = rng.choice(
    len(X),
    size=min(20_000, len(X)),
    replace=False
)

for k in k_values_detailed:

    print(f"K = {k}")

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X)

    silhouette = silhouette_score(
        X[silhouette_idx],
        labels[silhouette_idx],
        metric="cosine"
    )

    ari = adjusted_rand_score(y, labels)
    nmi = normalized_mutual_info_score(y, labels)

    results_detailed.append({
        "K": k,
        "Silhouette": silhouette,
        "ARI": ari,
        "NMI": nmi
    })

results_detailed_df = pd.DataFrame(results_detailed)

results_detailed_df

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Avenir Next"

ACCENT = "#6C63FF"

fig, ax = plt.subplots(figsize=(11, 5.5))

ax.plot(
    results_detailed_df["K"],
    results_detailed_df["Silhouette"],
    color=ACCENT,
    linewidth=2.5,
    marker="o",
    markersize=6,
    label="Silhouette"
)

ax.plot(
    results_detailed_df["K"],
    results_detailed_df["ARI"],
    color=ACCENT,
    linewidth=2.5,
    linestyle="--",
    marker="s",
    markersize=5,
    label="ARI"
)

ax.plot(
    results_detailed_df["K"],
    results_detailed_df["NMI"],
    color=ACCENT,
    linewidth=2.5,
    linestyle=":",
    marker="^",
    markersize=6,
    label="NMI"
)

ax.set_title(
    "Качество кластеризации при разных K",
    fontsize=20,
    fontweight="bold",
    pad=25
)

ax.set_xlabel(
    "Количество кластеров",
    fontsize=11,
    labelpad=12
)

ax.set_ylabel(
    "Значение метрики",
    fontsize=11,
    labelpad=12
)

ax.set_xticks(k_values_detailed)

ax.grid(
    axis="y",
    alpha=0.12,
    linewidth=0.8
)

ax.tick_params(
    axis="both",
    length=0,
    labelsize=10
)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.legend(
    frameon=False,
    loc="upper right"
)

plt.tight_layout()
plt.show()

In [ ]:
best_silhouette = results_detailed_df.loc[
    results_detailed_df["Silhouette"].idxmax()
]

best_ari = results_detailed_df.loc[
    results_detailed_df["ARI"].idxmax()
]

best_nmi = results_detailed_df.loc[
    results_detailed_df["NMI"].idxmax()
]

print("Лучший Silhouette:")
print(best_silhouette)

print("\nЛучший ARI:")
print(best_ari)

print("\nЛучший NMI:")
print(best_nmi)

Вывод по подробному эксперименту K-Means

На стратифицированной выборке из 100 000 товаров были исследованы значения числа кластеров K от 5 до 100. Качество кластеризации оценивалось по трём метрикам: Silhouette Score, ARI и NMI.

Основной результат — K = 10 является наиболее сбалансированным вариантом:

* Silhouette = 0.091 — максимальное значение среди всех рассмотренных K;
* ARI = 0.398 — также максимальное значение;
* NMI = 0.501 — немного ниже максимального значения, полученного при K = 50 (0.554).

При увеличении числа кластеров после K = 10 наблюдается общее снижение ARI:
K=10 → 0.398

K=15 → 0.389

K=20 → 0.357

K=25 → 0.318

K=30 → 0.283

K=50 → 0.237

K=100 → 0.144

Это означает, что увеличение количества кластеров приводит к более мелкому разбиению embedding-пространства, но полученные кластеры всё хуже соответствуют исходным категориям товаров.

При этом NMI ведёт себя иначе: его максимальное значение достигается при K = 50:

K=10 → NMI 0.501

K=25 → NMI 0.544

K=50 → NMI 0.554

Это показывает, что при большем количестве кластеров сохраняется больше информации об исходных категориях, однако эта информация распределяется между большим количеством небольших кластеров. Поэтому высокий NMI при K=50 не означает, что K=50 является лучшим вариантом по совокупности критериев.

HDBSCAN

In [ ]:
!pip install hdbscan

In [ ]:
import hdbscan
import numpy as np
import pandas as pd

from sklearn.preprocessing import normalize
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import normalize
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

HDBSCAN_SIZE = 50_000
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)

hdb_idx = rng.choice(
    len(X),
    size=HDBSCAN_SIZE,
    replace=False
)

X_hdb = X[hdb_idx]
y_hdb = y[hdb_idx]

print("X_hdb:", X_hdb.shape)
print("y_hdb:", y_hdb.shape)
print("Категорий:", len(np.unique(y_hdb)))

In [ ]:
X_hdb_norm = normalize(X_hdb, norm="l2")

print("Shape:", X_hdb_norm.shape)
print("Norm первого вектора:", np.linalg.norm(X_hdb_norm[0]))

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=100,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom"
)

hdb_labels = clusterer.fit_predict(X_hdb_norm)

print("Кластеризация завершена")

In [ ]:
unique_labels, counts = np.unique(
    hdb_labels,
    return_counts=True
)

n_clusters = np.sum(unique_labels != -1)
n_noise = np.sum(hdb_labels == -1)
noise_pct = n_noise / len(hdb_labels) * 100

print("Количество кластеров:", n_clusters)
print("Объектов:", len(hdb_labels))
print("Шум:", n_noise)
print(f"Доля шума: {noise_pct:.2f}%")

In [ ]:
cluster_sizes = (
    pd.Series(hdb_labels)
    .value_counts()
    .sort_index()
)

print(cluster_sizes)

In [ ]:
cluster_sizes_no_noise = (
    pd.Series(hdb_labels[hdb_labels != -1])
    .value_counts()
    .sort_values(ascending=False)
)

print(cluster_sizes_no_noise)